In [2]:
import pandas as pd

# ---- paths ----
BASELINE_PATH = "structured/metrics_long.csv"
DCPL_PATH = "runs/2026-01-13_121540/permodel_split80_dcpl_summary_gate-ridge.csv"

# ---- load baseline ----
base = pd.read_csv(BASELINE_PATH).rename(columns={"MRE(%)": "MRE_pct"})

# ---- load dcpl ----
dcpl = pd.read_csv(DCPL_PATH).rename(columns={"MRE(%)": "MRE_pct"})

# align naming: baseline uses model_tag; dcpl uses per_model_file (your sample shows it's like data_llama-7b etc.)
dcpl = dcpl.rename(columns={"per_model_file": "model_tag"})

# (optional) ensure learner name is consistent
# dcpl["learner"] is already "dcpl_gate=ridge" from your CSV
# but you can enforce:
dcpl["learner"] = dcpl["learner"].astype(str)

# choose only columns that exist in baseline metrics_long
keep_cols = ["experiment", "learner", "cv", "target", "model_tag",
             "n_train", "n_test", "R2", "MAE", "RMSE", "MRE_pct"]

dcpl = dcpl[keep_cols]

# ---- combine ----
df_all = pd.concat([base[keep_cols], dcpl], ignore_index=True)

# If you want to focus on throughput only (recommended for apples-to-apples)
df_all = df_all[df_all["target"] == "Target_throughput_tokens_per_sec"].copy()

df_all.head()


,experiment,learner,cv,target,model_tag,n_train,n_test,R2,MAE,RMSE,MRE_pct
0,baseline_split80,lr,split80_20,Target_throughput_tokens_per_sec,data_EleutherAI_gpt-neox-20b,71775,17944,0.075918,227.410480,561.537105,439.430353
1,baseline_split80,lr,split80_20,Target_throughput_tokens_per_sec,data_Salesforce_codegen2-16B,4820,1205,0.064251,162.835778,406.964038,1228.631199
2,baseline_split80,lr,split80_20,Target_throughput_tokens_per_sec,data_bigcode_starcoder,107817,26955,0.075314,240.540412,578.399803,290.389568
3,baseline_split80,lr,split80_20,Target_throughput_tokens_per_sec,data_bigscience_mt0-xxl,8564,2142,0.100527,236.010031,488.854214,507.497270
4,baseline_split80,lr,split80_20,Target_throughput_tokens_per_sec,data_google_flan-t5-xl,110639,27660,0.073391,165.963816,414.601455,463.190780


In [3]:
summary_all = (
    df_all.groupby("learner")[["R2", "MAE", "RMSE", "MRE_pct"]]
          .agg(["mean", "std"])
          .round(4)
)

summary_all


R2               MAE               RMSE            \
                   mean     std      mean      std      mean       std   
learner                                                                  
dcpl_gate=ridge  0.1346  0.0315  190.8965  54.5512  515.0762  143.4134   
llm_pilot        0.0111  0.0064  225.6158  63.8790  550.7919  153.5254   
lr               0.0763  0.0100  220.1139  62.6317  532.4424  148.9756   
nn               0.1293  0.0257  193.0238  55.4666  516.9158  144.9493   
rf_light         0.1344  0.0315  191.2186  54.7162  515.1473  143.4242   
ridge            0.0763  0.0100  220.1107  62.6317  532.4424  148.9756   

                  MRE_pct            
                     mean       std  
learner                              
dcpl_gate=ridge  183.7154   31.6616  
llm_pilot        823.2128  554.9158  
lr               580.3183  341.9733  
nn               193.2726   43.0405  
rf_light         188.0526   33.0800  
ridge            580.2649  341.8884

In [4]:
pivot_r2_all = (
    df_all.pivot_table(
        index="model_tag",
        columns="learner",
        values="R2",
        aggfunc="mean"
    )
    .round(4)
    .sort_index()
)

pivot_r2_all


learner,dcpl_gate=ridge,llm_pilot,lr,nn,rf_light,ridge
model_tag,,,,,,
data_EleutherAI_gpt-neox-20b,0.1153,0.0139,0.0759,0.1123,0.1150,0.0759
data_Salesforce_codegen2-16B,0.1677,-0.0001,0.0643,0.1465,0.1675,0.0643
data_bigcode_starcoder,0.1116,0.0205,0.0753,0.1096,0.1114,0.0753
data_bigscience_mt0-xxl,0.1787,0.0072,0.1005,0.1834,0.1788,0.1005
data_google_flan-t5-xl,0.1163,0.0118,0.0734,0.1184,0.1162,0.0734
data_google_flan-t5-xxl,0.1274,0.0092,0.0779,0.1274,0.1267,0.0779
data_google_flan-ul2,0.1073,0.0055,0.0661,0.1057,0.1067,0.0661
data_ibm_mpt-7b-instruct2,0.1906,0.0078,0.0813,0.1603,0.1900,0.0813
data_llama-13b,0.1181,0.0185,0.0761,0.1173,0.1184,0.0761


In [5]:
metrics = ["R2", "MAE", "RMSE", "MRE_pct"]

combined_all = (
    df_all.set_index(["model_tag", "learner"])[metrics]
          .unstack("learner")
          .round(4)
          .sort_index()
)

combined_all


R2                            \
learner                      dcpl_gate=ridge llm_pilot      lr      nn   
model_tag                                                                
data_EleutherAI_gpt-neox-20b          0.1153    0.0139  0.0759  0.1123   
data_Salesforce_codegen2-16B          0.1677   -0.0001  0.0643  0.1465   
data_bigcode_starcoder                0.1116    0.0205  0.0753  0.1096   
data_bigscience_mt0-xxl               0.1787    0.0072  0.1005  0.1834   
data_google_flan-t5-xl                0.1163    0.0118  0.0734  0.1184   
data_google_flan-t5-xxl               0.1274    0.0092  0.0779  0.1274   
data_google_flan-ul2                  0.1073    0.0055  0.0661  0.1057   
data_ibm_mpt-7b-instruct2             0.1906    0.0078  0.0813  0.1603   
data_llama-13b                        0.1181    0.0185  0.0761  0.1173   
data_llama-7b                         0.1133    0.0169  0.0718  0.1124   

                                                          MAE            \
learner                      rf_light   ridge dcpl_gate=ridge llm_pilot   
model_tag                                                                 
data_EleutherAI_gpt-neox-20b   0.1150  0.0759        209.6197  235.7329   
data_Salesforce_codegen2-16B   0.1675  0.0643        120.1618  167.8564   
data_bigcode_starcoder         0.1114  0.0753        220.4319  243.3180   
data_bigscience_mt0-xxl        0.1788  0.1005        200.0690  242.5048   
data_google_flan-t5-xl         0.1162  0.0734        148.0735  169.7300   
data_google_flan-t5-xxl        0.1267  0.0779        139.2006  161.0509   
data_google_flan-ul2           0.1067  0.0661        124.5751  142.7763   
data_ibm_mpt-7b-instruct2      0.1900  0.0813        248.3689  326.1499   
data_llama-13b                 0.1184  0.0761        222.2314  254.1890   
data_llama-7b                  0.1131  0.0718        276.2328  312.8501   

                                                  ...      RMSE            \
learner                             lr        nn  ...        lr        nn   
model_tag                                         ...                       
data_EleutherAI_gpt-neox-20b  227.4105  204.1043  ...  561.5371  550.3652   
data_Salesforce_codegen2-16B  162.8358  127.4660  ...  406.9640  388.6662   
data_bigcode_starcoder        240.5404  224.9447  ...  578.3998  567.5599   
data_bigscience_mt0-xxl       236.0100  202.0601  ...  488.8542  465.7843   
data_google_flan-t5-xl        165.9638  144.5941  ...  414.6015  404.4140   
data_google_flan-t5-xxl       156.0758  135.6982  ...  388.4963  377.9246   
data_google_flan-ul2          139.8003  128.0532  ...  347.2280  339.7776   
data_ibm_mpt-7b-instruct2     316.7611  260.6267  ...  739.1117  706.6373   
data_llama-13b                247.8297  229.6452  ...  623.3835  609.3305   
data_llama-7b                 307.9114  273.0459  ...  775.8482  758.6984   

                                                         MRE_pct             \
learner                       rf_light     ridge dcpl_gate=ridge  llm_pilot   
model_tag                                                                     
data_EleutherAI_gpt-neox-20b  549.5310  561.5371        176.2424   462.6393   
data_Salesforce_codegen2-16B  383.8667  406.9637        264.7814  1946.7850   
data_bigcode_starcoder        567.0102  578.3998        168.8678   340.4480   
data_bigscience_mt0-xxl       467.1114  488.8543        148.6060   746.0630   
data_google_flan-t5-xl        404.9102  414.6015        187.3526   689.6614   
data_google_flan-t5-xxl       378.0861  388.4963        194.4647   820.2815   
data_google_flan-ul2          339.5869  347.2280        184.7762   657.0933   
data_ibm_mpt-7b-instruct2     694.0244  739.1115        184.8678  1709.2590   
data_llama-13b                608.9657  623.3835        162.9655   437.3915   
data_llama-7b                 758.3800  775.8482        164.2302   422.5062   

                                                                        


In [6]:
# Split baseline vs dcpl
dcpl_only = df_all[df_all["learner"].str.contains("dcpl", case=False)].copy()
base_only = df_all[~df_all["learner"].str.contains("dcpl", case=False)].copy()

# Best baseline per model: R2 max, and errors min
best_base = base_only.groupby("model_tag").agg(
    base_best_R2=("R2", "max"),
    base_best_MAE=("MAE", "min"),
    base_best_RMSE=("RMSE", "min"),
    base_best_MRE=("MRE_pct", "min"),
).reset_index()

dcpl_vals = dcpl_only.groupby("model_tag").agg(
    dcpl_R2=("R2", "mean"),
    dcpl_MAE=("MAE", "mean"),
    dcpl_RMSE=("RMSE", "mean"),
    dcpl_MRE=("MRE_pct", "mean"),
).reset_index()

delta = best_base.merge(dcpl_vals, on="model_tag", how="inner")

delta["ΔR2"]   = delta["dcpl_R2"] - delta["base_best_R2"]
delta["ΔMAE"]  = delta["base_best_MAE"] - delta["dcpl_MAE"]     # positive means DCPL better (lower error)
delta["ΔRMSE"] = delta["base_best_RMSE"] - delta["dcpl_RMSE"]
delta["ΔMRE"]  = delta["base_best_MRE"] - delta["dcpl_MRE"]

delta = delta[["model_tag", "base_best_R2", "dcpl_R2", "ΔR2",
               "base_best_MAE", "dcpl_MAE", "ΔMAE",
               "base_best_RMSE", "dcpl_RMSE", "ΔRMSE",
               "base_best_MRE", "dcpl_MRE", "ΔMRE"]].round(4)

delta


,model_tag,base_best_R2,dcpl_R2,ΔR2,base_best_MAE,dcpl_MAE,ΔMAE,base_best_RMSE,dcpl_RMSE,ΔRMSE,base_best_MRE,dcpl_MRE,ΔMRE
0,data_EleutherAI_gpt-neox-20b,0.1150,0.1153,0.0003,204.1043,209.6197,-5.5154,549.5310,549.4504,0.0806,165.4973,176.2424,-10.7451
1,data_Salesforce_codegen2-16B,0.1675,0.1677,0.0003,120.8024,120.1618,0.6406,383.8667,383.8036,0.0632,276.5350,264.7814,11.7536
2,data_bigcode_starcoder,0.1114,0.1116,0.0002,220.4172,220.4319,-0.0147,567.0102,566.9431,0.0671,172.0224,168.8678,3.1546
3,data_bigscience_mt0-xxl,0.1834,0.1787,-0.0047,200.9645,200.0690,0.8954,465.7843,467.1200,-1.3358,150.6288,148.6060,2.0228
4,data_google_flan-t5-xl,0.1184,0.1163,-0.0021,144.5941,148.0735,-3.4794,404.4140,404.8989,-0.4850,171.2504,187.3526,-16.1022
5,data_google_flan-t5-xxl,0.1274,0.1274,-0.0000,135.6982,139.2006,-3.5024,377.9246,377.9253,-0.0007,186.8171,194.4647,-7.6476
6,data_google_flan-ul2,0.1067,0.1073,0.0005,125.7187,124.5751,1.1436,339.5869,339.4871,0.0999,187.2992,184.7762,2.5230
7,data_ibm_mpt-7b-instruct2,0.1900,0.1906,0.0006,249.5939,248.3689,1.2250,694.0244,693.7696,0.2548,194.5472,184.8678,9.6795
8,data_llama-13b,0.1184,0.1181,-0.0003,223.1184,222.2314,0.8870,608.9657,609.0678,-0.1022,168.9034,162.9655,5.9379
9,data_llama-7b,0.1131,0.1133,0.0002,273.0459,276.2328,-3.1870,758.3800,758.2961,0.0839,168.5705,164.2302,4.3403


In [8]:
import os
OUT_DIR = "results/structured/tables"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_FILTER = "Target_throughput_tokens_per_sec"  # change or set None to keep all
BASELINE_TAG = "baseline"  # only used for naming

# Learner ordering (optional; will keep only those present)
LEARNER_ORDER = ["lr", "ridge", "rf_light", "nn", "llm_pilot", "dcpl_gate=ridge"]

# =======================
# LOAD + MERGE
# =======================
base = pd.read_csv(BASELINE_PATH).rename(columns={"MRE(%)": "MRE_pct"})
dcpl = pd.read_csv(DCPL_PATH).rename(columns={"MRE(%)": "MRE_pct"}).rename(columns={"per_model_file": "model_tag"})

keep_cols = ["experiment", "learner", "cv", "target", "model_tag", "n_train", "n_test", "R2", "MAE", "RMSE", "MRE_pct"]

# Some baseline files may not have n_train/n_test; create if missing
for col in ["n_train", "n_test"]:
    if col not in base.columns:
        base[col] = pd.NA
base = base[keep_cols]

dcpl = dcpl[keep_cols]

df = pd.concat([base, dcpl], ignore_index=True)

if TARGET_FILTER is not None and "target" in df.columns:
    df = df[df["target"] == TARGET_FILTER].copy()

# Keep a consistent learner order (optional)
present = [l for l in LEARNER_ORDER if l in set(df["learner"].astype(str))]
if present:
    df["learner"] = pd.Categorical(df["learner"].astype(str), categories=present, ordered=True)

# =======================
# 1) SEPARATED METRICS TABLES (per metric)
# =======================
# Each table: rows=model_tag, columns=learner, value=mean(metric)
metrics = ["R2", "MAE", "RMSE", "MRE_pct"]

tables = {}
for m in metrics:
    t = (
        df.pivot_table(index="model_tag", columns="learner", values=m, aggfunc="mean")
          .round(4)
          .sort_index()
    )
    tables[m] = t
    out_csv = os.path.join(OUT_DIR, f"per_model_{m}.csv")
    t.to_csv(out_csv)

    # LaTeX version
    out_tex = os.path.join(OUT_DIR, f"per_model_{m}.tex")
    t.to_latex(out_tex, float_format="%.4f", escape=True)

print("[OK] Wrote per-metric tables to:", OUT_DIR)

# =======================
# 2) OPTIONAL: OVERALL SUMMARY (mean±std per learner)
# =======================
summary = (
    df.groupby("learner")[metrics]
      .agg(["mean", "std"])
      .round(4)
)
summary.to_csv(os.path.join(OUT_DIR, "summary_by_learner.csv"))
summary.to_latex(os.path.join(OUT_DIR, "summary_by_learner.tex"), float_format="%.4f", escape=True)

print("[OK] Wrote summary_by_learner.(csv|tex)")

# =======================
# 3) OPTIONAL: DELTA TABLE (DCPL vs best baseline per model)
# =======================
# Best baseline chosen per model per metric:
# - R2: max baseline
# - errors: min baseline
baseline_mask = ~df["learner"].astype(str).str.contains("dcpl", case=False, na=False)
dcpl_mask = df["learner"].astype(str).str.contains("dcpl", case=False, na=False)

base_only = df[baseline_mask].copy()
dcpl_only = df[dcpl_mask].copy()

if not base_only.empty and not dcpl_only.empty:
    best_base = base_only.groupby("model_tag").agg(
        base_best_R2=("R2", "max"),
        base_best_MAE=("MAE", "min"),
        base_best_RMSE=("RMSE", "min"),
        base_best_MRE=("MRE_pct", "min"),
    ).reset_index()

    dcpl_vals = dcpl_only.groupby("model_tag").agg(
        dcpl_R2=("R2", "mean"),
        dcpl_MAE=("MAE", "mean"),
        dcpl_RMSE=("RMSE", "mean"),
        dcpl_MRE=("MRE_pct", "mean"),
    ).reset_index()

    delta = best_base.merge(dcpl_vals, on="model_tag", how="inner")

    delta["ΔR2"]   = delta["dcpl_R2"] - delta["base_best_R2"]
    delta["ΔMAE"]  = delta["base_best_MAE"] - delta["dcpl_MAE"]
    delta["ΔRMSE"] = delta["base_best_RMSE"] - delta["dcpl_RMSE"]
    delta["ΔMRE"]  = delta["base_best_MRE"] - delta["dcpl_MRE"]

    delta = delta.round(4).sort_values("model_tag")

    delta_out_csv = os.path.join(OUT_DIR, "delta_dcpl_vs_best_baseline.csv")
    delta.to_csv(delta_out_csv, index=False)

    delta_out_tex = os.path.join(OUT_DIR, "delta_dcpl_vs_best_baseline.tex")
    delta.to_latex(delta_out_tex, index=False, float_format="%.4f", escape=True)

    print("[OK] Wrote delta_dcpl_vs_best_baseline.(csv|tex)")
else:
    print("[SKIP] Delta table not created (missing baseline or DCPL rows).")

# Preview one metric table in notebook
tables["R2"].head()

/var/folders/wv/4ptp8v2x7h775186_s12srf80000gs/T/ipykernel_42715/3250795704.py:46: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(index="model_tag", columns="learner", values=m, aggfunc="mean")


[OK] Wrote per-metric tables to: results/structured/tables
[OK] Wrote summary_by_learner.(csv|tex)
[OK] Wrote delta_dcpl_vs_best_baseline.(csv|tex)


/var/folders/wv/4ptp8v2x7h775186_s12srf80000gs/T/ipykernel_42715/3250795704.py:46: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(index="model_tag", columns="learner", values=m, aggfunc="mean")
/var/folders/wv/4ptp8v2x7h775186_s12srf80000gs/T/ipykernel_42715/3250795704.py:46: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(index="model_tag", columns="learner", values=m, aggfunc="mean")
/var/folders/wv/4ptp8v2x7h775186_s12srf80000gs/T/ipykernel_42715/3250795704.py:46: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warni

learner,lr,ridge,rf_light,nn,llm_pilot,dcpl_gate=ridge
model_tag,,,,,,
data_EleutherAI_gpt-neox-20b,0.0759,0.0759,0.1150,0.1123,0.0139,0.1153
data_Salesforce_codegen2-16B,0.0643,0.0643,0.1675,0.1465,-0.0001,0.1677
data_bigcode_starcoder,0.0753,0.0753,0.1114,0.1096,0.0205,0.1116
data_bigscience_mt0-xxl,0.1005,0.1005,0.1788,0.1834,0.0072,0.1787
data_google_flan-t5-xl,0.0734,0.0734,0.1162,0.1184,0.0118,0.1163


Merge results

In [16]:
from pathlib import Path
import pandas as pd

# =========================
# CONFIG: EDIT THESE PATHS IF NEEDED
# =========================
BASELINE_STACKED = Path(
    "runs/20260129_090459/30x_baselines/"
    "baseline_split80__MULTI_BASELINES__30x_base42/"
    "baseline_split80_permodel_ALL_learners_ALL_runs_stacked.csv"
)

DICE_FILES = {
    # keys here are the *normalized* learner names we want in the output
    "lr": Path(
        "runs/20260129_023223/dice_runs/"
        "dice_split80__lr__multirun_30x_base42/"
        "dice_split80_permodel_mean_across_runs.csv"
    ),
    "nn": Path(
        "runs/20260129_023223/dice_runs/"
        "dice_split80__nn__multirun_30x_base42/"
        "dice_split80_permodel_mean_across_runs.csv"
    ),
    "rf_light": Path(  # your DICE folder name is rf, but we normalize to rf_light for comparison
        "runs/20260129_023223/dice_runs/"
        "dice_split80__rf__multirun_30x_base42/"
        "dice_split80_permodel_mean_across_runs.csv"
    ),
    "ridge": Path(
        "runs/20260129_023223/dice_runs/"
        "dice_split80__ridge__multirun_30x_base42/"
        "dice_split80_permodel_mean_across_runs.csv"  # ✅ FIXED underscore
    ),
}

OUT_DIR = Path("results/latex_tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "mean_baseline_vs_dice_selected.csv"

LEARNERS_KEEP = {"lr", "ridge", "rf_light", "nn"}
METRICS = ["R2", "MAE", "RMSE", "MRE"]

# =========================
# HELPERS
# =========================
def _must_exist(p: Path) -> None:
    if not p.exists():
        raise FileNotFoundError(f"File not found:\n  {p}\n(Current working dir: {Path.cwd()})")

def _require_cols(df: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing required columns: {missing}\nAvailable: {list(df.columns)}")

# =========================
# 1) BASELINE: compute mean over 30 runs
# =========================
print("[LOAD] Baseline stacked runs:")
print(" ", BASELINE_STACKED)
_must_exist(BASELINE_STACKED)

df_base = pd.read_csv(BASELINE_STACKED)

_required_base_cols = ["per_model_file", "learner"] + METRICS
_require_cols(df_base, _required_base_cols, "baseline_stacked")

# filter selected learners
df_base = df_base[df_base["learner"].isin(LEARNERS_KEEP)].copy()

# compute mean across runs
base_mean = (
    df_base
    .groupby(["per_model_file", "target", "cv", "learner"], as_index=False)[METRICS]
    .mean()
)
base_mean["framework"] = "baseline"

# =========================
# 2) DICE: already mean across runs
# =========================
dice_frames = []

for learner_norm, csv_path in DICE_FILES.items():
    print(f"[LOAD] DICE mean for learner={learner_norm}:")
    print(" ", csv_path)
    _must_exist(csv_path)

    df = pd.read_csv(csv_path)

    _require_cols(df, ["per_model_file", "learner"] + METRICS, f"dice_{learner_norm}")

    # keep only dice rows (e.g., dice_lr)
    df = df[df["learner"].astype(str).str.startswith("dice_")].copy()

    # normalize learner name to match baseline naming
    df["learner"] = learner_norm
    df["framework"] = "dice"

    # ensure target/cv exist for clean merge
    if "target" not in df.columns:
        df["target"] = None
    if "cv" not in df.columns:
        df["cv"] = "split80_20"

    dice_frames.append(df[["per_model_file", "target", "cv", "learner", "framework"] + METRICS])

dice_mean = pd.concat(dice_frames, ignore_index=True)

# =========================
# 3) MERGE + SAVE
# =========================
final_df = pd.concat(
    [
        base_mean[["per_model_file", "target", "cv", "learner", "framework"] + METRICS],
        dice_mean,
    ],
    ignore_index=True,
)

final_df = final_df.sort_values(["per_model_file", "framework", "learner"]).reset_index(drop=True)

final_df.to_csv(OUT_CSV, index=False)

print("\n[DONE]")
print(f"Saved: {OUT_CSV}")
print(f"Rows : {len(final_df)}")


[LOAD] Baseline stacked runs:
  runs/20260129_090459/30x_baselines/baseline_split80__MULTI_BASELINES__30x_base42/baseline_split80_permodel_ALL_learners_ALL_runs_stacked.csv
[LOAD] DICE mean for learner=lr:
  runs/20260129_023223/dice_runs/dice_split80__lr__multirun_30x_base42/dice_split80_permodel_mean_across_runs.csv
[LOAD] DICE mean for learner=nn:
  runs/20260129_023223/dice_runs/dice_split80__nn__multirun_30x_base42/dice_split80_permodel_mean_across_runs.csv
[LOAD] DICE mean for learner=rf_light:
  runs/20260129_023223/dice_runs/dice_split80__rf__multirun_30x_base42/dice_split80_permodel_mean_across_runs.csv
[LOAD] DICE mean for learner=ridge:
  runs/20260129_023223/dice_runs/dice_split80__ridge__multirun_30x_base42/dice_split80_permodel_mean_across_runs.csv

[DONE]
Saved: results/latex_tables/mean_baseline_vs_dice_selected.csv
Rows : 80


In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
MERGED_CSV = Path("results/latex_tables/mean_baseline_vs_dice_selected.csv")

OUT_PLOTS_DIR = Path("results/latex_tables/plots")
OUT_TABLES_DIR = Path("results/latex_tables/tables")
OUT_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

LEARNERS = ["lr", "ridge", "rf_light", "nn"]
METRICS_TO_PLOT = ["R2", "MRE"]  # change/add: ["R2","MAE","RMSE","MRE"]
DATASET_COL = "per_model_file"

# If you want a stable x-axis order:
SORT_DATASETS_ALPHA = True  # True = alphabetical, False = keep original order

# =========================
# HELPERS
# =========================
def latex_escape(s: str) -> str:
    """Escape LaTeX special chars in text fields."""
    if s is None:
        return ""
    s = str(s)
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    return s

def require_cols(df: pd.DataFrame, cols: list[str], name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] missing columns: {missing}\nAvailable: {list(df.columns)}")

def format_float(x, nd=3):
    if pd.isna(x):
        return ""
    return f"{float(x):.{nd}f}"

def make_latex_table(df_rows: pd.DataFrame, caption: str, label: str) -> str:
    """
    Expects columns: Dataset, Baseline, DICE, Delta, DeltaPct
    """
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\begin{tabular}{lrrrr}")
    lines.append(r"\toprule")
    lines.append(r"Dataset & Baseline & DICE & $\Delta$ & $\%\Delta$ \\")
    lines.append(r"\midrule")

    for _, r in df_rows.iterrows():
        ds = latex_escape(r["Dataset"])
        b = r["Baseline"]
        d = r["DICE"]
        dl = r["Delta"]
        dp = r["DeltaPct"]

        lines.append(
            f"{ds} & {b} & {d} & {dl} & {dp} \\\\"
        )

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)

# =========================
# LOAD
# =========================
if not MERGED_CSV.exists():
    raise FileNotFoundError(
        f"Cannot find merged CSV:\n  {MERGED_CSV}\n"
        "Make sure you ran the merge script first."
    )

df = pd.read_csv(MERGED_CSV)
require_cols(df, [DATASET_COL, "framework", "learner"], "merged")
for m in METRICS_TO_PLOT:
    require_cols(df, [m], "merged")

# Keep only baseline & dice
df = df[df["framework"].isin(["baseline", "dice"])].copy()
df["learner"] = df["learner"].astype(str).str.strip().str.lower()

# =========================
# 1) PLOTS
# =========================
for learner in LEARNERS:
    df_l = df[df["learner"] == learner].copy()
    if df_l.empty:
        print(f"[WARN] No rows for learner={learner}. Skipping.")
        continue

    # Ensure both frameworks exist
    have_baseline = (df_l["framework"] == "baseline").any()
    have_dice = (df_l["framework"] == "dice").any()
    if not (have_baseline and have_dice):
        print(f"[WARN] learner={learner} missing baseline or dice. Skipping plots.")
        continue

    # pivot per metric
    for metric in METRICS_TO_PLOT:
        piv = (
            df_l.pivot_table(
                index=DATASET_COL,
                columns="framework",
                values=metric,
                aggfunc="mean",
            )
            .reset_index()
        )

        # drop datasets missing either baseline or dice
        piv = piv.dropna(subset=["baseline", "dice"]).copy()

        if SORT_DATASETS_ALPHA:
            piv = piv.sort_values(DATASET_COL).reset_index(drop=True)

        datasets = piv[DATASET_COL].astype(str).tolist()
        x = np.arange(len(datasets))

        y_base = piv["baseline"].to_numpy(dtype=float)
        y_dice = piv["dice"].to_numpy(dtype=float)

        plt.figure(figsize=(max(10, len(datasets) * 0.8), 5))

        # offset kecil agar baseline & dice tidak menimpa
        plt.plot(
            x - 0.05, y_base,
            marker="o",
            linestyle="-",
            linewidth=2,
            label=f"{learner} baseline",
        )
        plt.plot(
            x + 0.05, y_dice,
            marker="o",
            linestyle="--",
            linewidth=2,
            label=f"{learner} dice",
        )

        plt.title(f"Baseline vs DICE — learner={learner} — metric={metric}")
        plt.xlabel("Dataset")
        plt.ylabel(metric)

        plt.xticks(
            x,
            [d.replace("data_", "") for d in datasets],  # optional: rapikan nama
            rotation=45,
            ha="right",
            fontsize=9,
        )

        plt.legend()
        plt.tight_layout()


        out_png = OUT_PLOTS_DIR / f"line_{learner}_{metric}.png"
        plt.tight_layout()
        plt.savefig(out_png, dpi=200)
        plt.close()

        # Also save dataset mapping for the x-axis
        mapping_csv = OUT_PLOTS_DIR / f"line_{learner}_{metric}_xmap.csv"
        piv[[DATASET_COL]].assign(dataset_index=x).to_csv(mapping_csv, index=False)

        print(f"[OK] plot saved: {out_png}")
        print(f"[OK] x-map saved: {mapping_csv}")

# =========================
# 2) LATEX TABLES (per learner x metric)
# =========================
for learner in LEARNERS:
    df_l = df[df["learner"] == learner].copy()
    if df_l.empty:
        continue

    for metric in METRICS_TO_PLOT:
        piv = (
            df_l.pivot_table(
                index=DATASET_COL,
                columns="framework",
                values=metric,
                aggfunc="mean",
            )
            .reset_index()
        )
        piv = piv.dropna(subset=["baseline", "dice"]).copy()

        # improvement
        piv["delta"] = piv["dice"] - piv["baseline"]
        piv["delta_pct"] = np.where(
            np.abs(piv["baseline"].to_numpy(dtype=float)) < 1e-12,
            np.nan,
            100.0 * (piv["delta"].to_numpy(dtype=float) / piv["baseline"].to_numpy(dtype=float)),
        )

        # For MRE/MAE/RMSE, lower is better — so "improvement" is negative delta.
        # We still show delta as (dice-baseline), and sort by "best" automatically:
        if metric in ["MAE", "RMSE", "MRE"]:
            # best = most negative delta (largest reduction)
            piv = piv.sort_values("delta", ascending=True)
        else:
            # for R2, best = largest positive delta
            piv = piv.sort_values("delta", ascending=False)

        out_rows = pd.DataFrame({
            "Dataset": piv[DATASET_COL].astype(str),
            "Baseline": piv["baseline"].map(lambda v: format_float(v, 3)),
            "DICE": piv["dice"].map(lambda v: format_float(v, 3)),
            "Delta": piv["delta"].map(lambda v: format_float(v, 3)),
            "DeltaPct": piv["delta_pct"].map(lambda v: ("" if pd.isna(v) else f"{float(v):.2f}\\%")),
        })

        caption = f"Baseline vs DICE (learner={learner}) on {metric} (mean over runs). " \
                  f"$\\Delta$ = DICE - Baseline."
        label = f"tab:baseline_vs_dice_{learner}_{metric}".replace("_", "-")

        tex = make_latex_table(out_rows, caption=caption, label=label)

        out_tex = OUT_TABLES_DIR / f"table_{learner}_{metric}.tex"
        out_tex.write_text(tex, encoding="utf-8")

        # also save a CSV companion (useful for debugging)
        out_csv = OUT_TABLES_DIR / f"table_{learner}_{metric}.csv"
        out_rows.to_csv(out_csv, index=False)

        print(f"[OK] latex saved: {out_tex}")
        print(f"[OK] table csv saved: {out_csv}")

print("\n[DONE] All plots + LaTeX tables generated.")
print(f"Plots : {OUT_PLOTS_DIR}")
print(f"Tables: {OUT_TABLES_DIR}")


[OK] plot saved: results/latex_tables/plots/line_lr_R2.png
[OK] x-map saved: results/latex_tables/plots/line_lr_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_lr_MRE.png
[OK] x-map saved: results/latex_tables/plots/line_lr_MRE_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_ridge_R2.png
[OK] x-map saved: results/latex_tables/plots/line_ridge_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_ridge_MRE.png
[OK] x-map saved: results/latex_tables/plots/line_ridge_MRE_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_rf_light_R2.png
[OK] x-map saved: results/latex_tables/plots/line_rf_light_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_rf_light_MRE.png
[OK] x-map saved: results/latex_tables/plots/line_rf_light_MRE_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_nn_R2.png
[OK] x-map saved: results/latex_tables/plots/line_nn_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_nn_MRE.png
[OK] x-map saved: results/late

In [18]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
MERGED_CSV = Path("results/latex_tables/mean_baseline_vs_dice_selected.csv")

OUT_PLOTS_DIR = Path("results/latex_tables/plots")
OUT_TABLES_DIR = Path("results/latex_tables/tables")
OUT_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

LEARNERS = ["lr", "ridge", "rf_light", "nn"]
METRICS_TO_PLOT = ["R2", "MRE"]  # change/add: ["R2","MAE","RMSE","MRE"]
DATASET_COL = "per_model_file"

# If you want a stable x-axis order:
SORT_DATASETS_ALPHA = True  # True = alphabetical, False = keep original order

# =========================
# HELPERS
# =========================
def latex_escape(s: str) -> str:
    """Escape LaTeX special chars in text fields."""
    if s is None:
        return ""
    s = str(s)
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    return s

def require_cols(df: pd.DataFrame, cols: list[str], name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] missing columns: {missing}\nAvailable: {list(df.columns)}")

def format_float(x, nd=3):
    if pd.isna(x):
        return ""
    return f"{float(x):.{nd}f}"

def make_latex_table(df_rows: pd.DataFrame, caption: str, label: str) -> str:
    """
    Expects columns: Dataset, Baseline, DICE, Delta, DeltaPct
    """
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\begin{tabular}{lrrrr}")
    lines.append(r"\toprule")
    lines.append(r"Dataset & Baseline & DICE & $\Delta$ & $\%\Delta$ \\")
    lines.append(r"\midrule")

    for _, r in df_rows.iterrows():
        ds = latex_escape(r["Dataset"])
        b = r["Baseline"]
        d = r["DICE"]
        dl = r["Delta"]
        dp = r["DeltaPct"]

        lines.append(
            f"{ds} & {b} & {d} & {dl} & {dp} \\\\"
        )

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)

# =========================
# LOAD
# =========================
if not MERGED_CSV.exists():
    raise FileNotFoundError(
        f"Cannot find merged CSV:\n  {MERGED_CSV}\n"
        "Make sure you ran the merge script first."
    )

df = pd.read_csv(MERGED_CSV)
require_cols(df, [DATASET_COL, "framework", "learner"], "merged")
for m in METRICS_TO_PLOT:
    require_cols(df, [m], "merged")

# Keep only baseline & dice
df = df[df["framework"].isin(["baseline", "dice"])].copy()
df["learner"] = df["learner"].astype(str).str.strip().str.lower()

# =========================
# 1) PLOTS
# =========================
for learner in LEARNERS:
    df_l = df[df["learner"] == learner].copy()
    if df_l.empty:
        print(f"[WARN] No rows for learner={learner}. Skipping.")
        continue

    # Ensure both frameworks exist
    have_baseline = (df_l["framework"] == "baseline").any()
    have_dice = (df_l["framework"] == "dice").any()
    if not (have_baseline and have_dice):
        print(f"[WARN] learner={learner} missing baseline or dice. Skipping plots.")
        continue

    # pivot per metric
    for metric in METRICS_TO_PLOT:
        piv = (
            df_l.pivot_table(
                index=DATASET_COL,
                columns="framework",
                values=metric,
                aggfunc="mean",
            )
            .reset_index()
        )

        # drop datasets missing either baseline or dice
        piv = piv.dropna(subset=["baseline", "dice"]).copy()

        if SORT_DATASETS_ALPHA:
            piv = piv.sort_values(DATASET_COL).reset_index(drop=True)

        datasets = piv[DATASET_COL].astype(str).tolist()
        x = np.arange(len(datasets))

        y_base = piv["baseline"].to_numpy(dtype=float)
        y_dice = piv["dice"].to_numpy(dtype=float)

        plt.figure(figsize=(max(10, len(datasets) * 0.8), 5))

        # offset kecil agar baseline & dice tidak menimpa
        plt.plot(
            x - 0.05, y_base,
            marker="o",
            linestyle="-",
            linewidth=2,
            label=f"{learner} baseline",
        )
        plt.plot(
            x + 0.05, y_dice,
            marker="o",
            linestyle="--",
            linewidth=2,
            label=f"{learner} dice",
        )

        plt.title(f"Baseline vs DICE — learner={learner} — metric={metric}")
        plt.xlabel("Dataset")
        plt.ylabel(metric)

        plt.xticks(
            x,
            [d.replace("data_", "") for d in datasets],  # optional: rapikan nama
            rotation=45,
            ha="right",
            fontsize=9,
        )

        plt.legend()
        plt.tight_layout()


        out_png = OUT_PLOTS_DIR / f"line_{learner}_{metric}.png"
        plt.tight_layout()
        plt.savefig(out_png, dpi=200)
        plt.close()

        # Also save dataset mapping for the x-axis
        mapping_csv = OUT_PLOTS_DIR / f"line_{learner}_{metric}_xmap.csv"
        piv[[DATASET_COL]].assign(dataset_index=x).to_csv(mapping_csv, index=False)

        print(f"[OK] plot saved: {out_png}")
        print(f"[OK] x-map saved: {mapping_csv}")

# =========================
# 2) LATEX TABLES (per learner x metric)
# =========================
# for learner in LEARNERS:
#     df_l = df[df["learner"] == learner].copy()
#     if df_l.empty:
#         continue

#     for metric in METRICS_TO_PLOT:
#         piv = (
#             df_l.pivot_table(
#                 index=DATASET_COL,
#                 columns="framework",
#                 values=metric,
#                 aggfunc="mean",
#             )
#             .reset_index()
#         )
#         piv = piv.dropna(subset=["baseline", "dice"]).copy()

#         # improvement
#         piv["delta"] = piv["dice"] - piv["baseline"]
#         piv["delta_pct"] = np.where(
#             np.abs(piv["baseline"].to_numpy(dtype=float)) < 1e-12,
#             np.nan,
#             100.0 * (piv["delta"].to_numpy(dtype=float) / piv["baseline"].to_numpy(dtype=float)),
#         )

#         # For MRE/MAE/RMSE, lower is better — so "improvement" is negative delta.
#         # We still show delta as (dice-baseline), and sort by "best" automatically:
#         if metric in ["MAE", "RMSE", "MRE"]:
#             # best = most negative delta (largest reduction)
#             piv = piv.sort_values("delta", ascending=True)
#         else:
#             # for R2, best = largest positive delta
#             piv = piv.sort_values("delta", ascending=False)

#         out_rows = pd.DataFrame({
#             "Dataset": piv[DATASET_COL].astype(str),
#             "Baseline": piv["baseline"].map(lambda v: format_float(v, 3)),
#             "DICE": piv["dice"].map(lambda v: format_float(v, 3)),
#             "Delta": piv["delta"].map(lambda v: format_float(v, 3)),
#             "DeltaPct": piv["delta_pct"].map(lambda v: ("" if pd.isna(v) else f"{float(v):.2f}\\%")),
#         })

#         caption = f"Baseline vs DICE (learner={learner}) on {metric} (mean over runs). " \
#                   f"$\\Delta$ = DICE - Baseline."
#         label = f"tab:baseline_vs_dice_{learner}_{metric}".replace("_", "-")

#         tex = make_latex_table(out_rows, caption=caption, label=label)

#         out_tex = OUT_TABLES_DIR / f"table_{learner}_{metric}.tex"
#         out_tex.write_text(tex, encoding="utf-8")

#         # also save a CSV companion (useful for debugging)
#         out_csv = OUT_TABLES_DIR / f"table_{learner}_{metric}.csv"
#         out_rows.to_csv(out_csv, index=False)

#         print(f"[OK] latex saved: {out_tex}")
#         print(f"[OK] table csv saved: {out_csv}")

print("\n[DONE] All plots + LaTeX tables generated.")
print(f"Plots : {OUT_PLOTS_DIR}")
# print(f"Tables: {OUT_TABLES_DIR}")


[OK] plot saved: results/latex_tables/plots/line_lr_R2.png
[OK] x-map saved: results/latex_tables/plots/line_lr_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_lr_MRE.png
[OK] x-map saved: results/latex_tables/plots/line_lr_MRE_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_ridge_R2.png
[OK] x-map saved: results/latex_tables/plots/line_ridge_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_ridge_MRE.png
[OK] x-map saved: results/latex_tables/plots/line_ridge_MRE_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_rf_light_R2.png
[OK] x-map saved: results/latex_tables/plots/line_rf_light_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_rf_light_MRE.png
[OK] x-map saved: results/latex_tables/plots/line_rf_light_MRE_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_nn_R2.png
[OK] x-map saved: results/latex_tables/plots/line_nn_R2_xmap.csv
[OK] plot saved: results/latex_tables/plots/line_nn_MRE.png
[OK] x-map saved: results/late